# TP 2: Text Processing, Bag-of-Words and N-gram Language Models

In this TP you build, in Python, the text-processing pipeline that turns raw text into index terms, then use it to build **bag-of-words** vectors, an **inverted index** and a **bigram language model**.

| Exercise | Topic | Points |
|---|---|---|
| 1 | Tokenization with regular expressions | 15 |
| 2 | Normalization, stop words, stemming and lemmatization | 25 |
| 3 | Subword tokenization (Hugging Face) | 10 |
| 4 | Bag-of-words and the inverted index | 20 |
| 5 | Bigram language model, add-k smoothing, perplexity | 30 |

* Complete every cell marked `# TODO`, and write short answers in the **Answer** cells.
* Submit **only this notebook**, renamed `TP2_StudentID_Name_Group.ipynb` for example `TP2_e202000129_DORN_Dawin_I5AMS_A.ipynb`, by email (my email: dorn.dawin@gmail.com) **[25/05/2026 11:59 PM]**.

## 0 · Setup
Run these two cells first. On Google Colab everything is already installed; the first cell only downloads NLTK word lists.

In [1]:
import nltk
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

import re, math
from collections import Counter
import pandas as pd

In [2]:
# Our small collection: 12 short documents about search engines
DOCS = [
    "Search engines index billions of web pages every day.",
    "A search engine ranks the indexed pages for each query.",
    "Users type short queries into the search box.",
    "The crawler downloads web pages and follows their links.",
    "Google's crawler found 3 new pages in the U.S.A. yesterday.",
    "Indexing makes searching fast, but building the index takes time.",
    "The inverted index maps each term to the documents that contain it.",
    "Users don't read every result; they click the top-ranked pages.",
    "Ranking models compare the query with each indexed document.",
    "Stemming maps searching, searched and searches to one term.",
    "Stop words such as 'the' and 'of' are very frequent.",
    "Language models assign a probability to every query.",
]
for i, d in enumerate(DOCS):
    print(i, d)

0 Search engines index billions of web pages every day.
1 A search engine ranks the indexed pages for each query.
2 Users type short queries into the search box.
3 The crawler downloads web pages and follows their links.
4 Google's crawler found 3 new pages in the U.S.A. yesterday.
5 Indexing makes searching fast, but building the index takes time.
6 The inverted index maps each term to the documents that contain it.
7 Users don't read every result; they click the top-ranked pages.
8 Ranking models compare the query with each indexed document.
9 Stemming maps searching, searched and searches to one term.
10 Stop words such as 'the' and 'of' are very frequent.
11 Language models assign a probability to every query.


## Exercise 1 · Tokenization (15 pts)
**Tokenization** splits a text into tokens (words, numbers, punctuation marks). It is the first step of the text transformation component of a search engine (Lecture 1).

### 1.1 · Splitting on spaces (3 pts)
Run the cell. Python's `str.split()` splits on whitespace only.

In [3]:
print(DOCS[4].split())
print(DOCS[7].split())

["Google's", 'crawler', 'found', '3', 'new', 'pages', 'in', 'the', 'U.S.A.', 'yesterday.']
['Users', "don't", 'read', 'every', 'result;', 'they', 'click', 'the', 'top-ranked', 'pages.']


**Question 1.1.** Name two problems you see in this output.

**Answer 1.1:** (1) Punctuation stays glued to the neighbouring word: `yesterday.`, `pages.`, `result;` and `'the'` are different strings from `yesterday`, `pages`, `result`, `the`, so the same word would get several index entries. (2) Case and clitics are not handled: `Users`/`users` stay distinct and `Google's` and `don't` are kept as single, odd tokens. (Spaces alone also cannot tell the abbreviation `U.S.A.` from a sentence end.)

### 1.2 · A regular-expression tokenizer (6 pts)
The pattern below is adapted from the NLTK example in Jurafsky & Martin, *Speech and Language Processing*, section 2.8. The alternatives are tried **from left to right**.

Complete `tokenize(text)` so that it returns the list of all matches of `PATTERN` in `text` (hint: `re.findall`).

In [4]:
PATTERN = r"""(?x)
    (?:[A-Z]\.)+            # abbreviations: U.S.A.
  | \w+(?:[-']\w+)*         # words, with optional internal hyphen or apostrophe
  | \$?\d+(?:\.\d+)?%?      # numbers, money, percentages
  | \.\.\.                  # ellipsis
  | [][.,;"'?():_`-]        # single punctuation marks
"""

def tokenize(text):
    return re.findall(PATTERN, text)

for i in (4, 7, 10):
    print(tokenize(DOCS[i]))

["Google's", 'crawler', 'found', '3', 'new', 'pages', 'in', 'the', 'U.S.A.', 'yesterday', '.']
['Users', "don't", 'read', 'every', 'result', ';', 'they', 'click', 'the', 'top-ranked', 'pages', '.']
['Stop', 'words', 'such', 'as', "'", 'the', "'", 'and', "'", 'of', "'", 'are', 'very', 'frequent', '.']


**Question 1.2.** (a) Compare with 1.1: what is fixed? (b) Why must the abbreviation alternative `(?:[A-Z]\.)+` come **before** the word alternative `\w+…`? (Try swapping them.)

**Answer 1.2:** (a) Final punctuation is now a separate token (`yesterday`, `.`), the quotes around `'the'` are split off, while `U.S.A.`, `Google's`, `don't` and `top-ranked` stay in one piece. (b) `re.findall` scans left to right and, at each position, takes the **first alternative that matches**. If `\w+…` came first, it would match just `U` in `U.S.A.`, then `.` would be matched as punctuation, then `S`, `.`, `A`, `.`, giving 6 tokens instead of one abbreviation. Putting the more specific pattern first prevents this.

### 1.3 · Tokens and types (6 pts)
* A **token** is one occurrence of a word; a **type** is a distinct word (an entry of the vocabulary).
* Tokenize all 12 documents, lower-case every token, and keep only tokens that contain a letter or a digit (drop pure punctuation).
* Print the number of tokens N, the number of types |V|, and the 10 most frequent tokens (`Counter.most_common`).

In [5]:
tokens = [t.lower() for d in DOCS for t in tokenize(d) if re.search(r"[A-Za-z0-9]", t)]
N = len(tokens)
V = len(set(tokens))
print("N =", N, " |V| =", V)
print(Counter(tokens).most_common(10))

N = 114  |V| = 77
[('the', 10), ('pages', 5), ('search', 3), ('index', 3), ('every', 3), ('each', 3), ('query', 3), ('and', 3), ('to', 3), ('of', 2)]


**Question 1.3.** Which kind of word is the most frequent? Why is that a problem for an index?

**Answer 1.3:** The most frequent token is `the` (10 times), followed by other function words (`and`, `to`, `each`, `every`, `of`). These words appear in almost every document, so they say nothing about what a document is about, yet they create the longest postings lists: they waste index space and query time and add noise to ranking.

## Exercise 2 · Normalization, stop words, stemming (25 pts)
**Normalization** maps tokens that should match to the same term, e.g. *Search*, *search* → *search*. **Stemming** and **lemmatization** reduce word variants (*searching*, *searches*) to a common form.

### 2.1 · Normalization (5 pts)
Complete `normalize(tokens)` with these rules, in this order:
1. lower-case every token;
2. drop tokens that contain no letter and no digit (punctuation);
3. remove a final possessive `'s`: `google's` → `google`;
4. remove the dots of abbreviations: `u.s.a.` → `usa`.

**Check:** `['google', 'crawler', 'found', '3', 'new', 'pages', 'in', 'the', 'usa', 'yesterday']`

In [6]:
def normalize(tokens):
    out = []
    for t in tokens:
        t = t.lower()                                  # rule 1
        if not re.search(r"[a-z0-9]", t):              # rule 2: pure punctuation
            continue
        if t.endswith("'s"):                           # rule 3: possessive
            t = t[:-2]
        if re.fullmatch(r"(?:[a-z]\.)+", t):           # rule 4: abbreviation u.s.a. -> usa
            t = t.replace(".", "")
        out.append(t)
    return out

print(normalize(tokenize(DOCS[4])))

['google', 'crawler', 'found', '3', 'new', 'pages', 'in', 'the', 'usa', 'yesterday']


### 2.2 · Stop words (5 pts)
Use NLTK's English stop list. Remove stop words from the normalized tokens of the **whole collection** and report the number of tokens before and after, and the percentage removed.

In [7]:
from nltk.corpus import stopwords
STOP = set(stopwords.words("english"))
print(len(STOP), "stop words, e.g.", sorted(STOP)[:10])

all_terms = [t for d in DOCS for t in normalize(tokenize(d))]   # normalized tokens of all documents
kept = [t for t in all_terms if t not in STOP]                  # without stop words
print(len(all_terms), "->", len(kept), f"({100 * (1 - len(kept) / len(all_terms)):.1f}% removed)")

198 stop words, e.g. ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an']
114 -> 77 (32.5% removed)


**Question 2.2.** Give one search query that would be damaged by removing stop words, and explain why.

**Answer 2.2:** The query *"to be or not to be"* (or the band *"The Who"*, or *"IT"*) is made only of stop words, so after removal nothing is left and the query cannot be answered. More generally, stop words carry meaning in phrases (*"flights to Paris"* vs *"flights from Paris"*), which is why modern engines often keep them.

### 2.3 · Porter vs Snowball vs lemmatization (8 pts)
Fill a table with one row per word and the columns: `porter`, `snowball`, `lemma_noun` (WordNet lemmatizer, default part of speech = noun) and `lemma_verb` (`pos="v"`).

In [8]:
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
porter, snowball, wnl = PorterStemmer(), SnowballStemmer("english"), WordNetLemmatizer()

WORDS = ["searching", "searched", "searches", "indexing", "engines", "queries", "users",
         "libraries", "generously", "university", "universe", "organization", "organ",
         "running", "feet", "was"]

rows = []
for w in WORDS:
    rows.append({"word": w,
                 "porter": porter.stem(w),
                 "snowball": snowball.stem(w),
                 "lemma_noun": wnl.lemmatize(w),
                 "lemma_verb": wnl.lemmatize(w, pos="v")})
pd.DataFrame(rows)

,word,porter,snowball,lemma_noun,lemma_verb
0,searching,search,search,searching,search
1,searched,search,search,searched,search
2,searches,search,search,search,search
3,indexing,index,index,indexing,index
4,engines,engin,engin,engine,engines
5,queries,queri,queri,query,query
6,users,user,user,user,users
7,libraries,librari,librari,library,libraries
8,generously,gener,generous,generously,generously
9,university,univers,univers,university,university


**Question 2.3.** (a) Give two stems that are not real English words. (b) Give one pair of words with **different meanings** that Porter maps to the same stem, and explain why this hurts precision. (c) Why does `wnl.lemmatize("searching")` return *searching*, but `pos="v"` returns *search*? (d) Give one word where Porter and Snowball disagree.

**Answer 2.3:** (a) Stems are not always words: `engin` (engines), `queri` (queries), `librari` (libraries), `univers`, `gener`, `wa` (was). (b) Porter maps both `university` and `universe` to `univers`, and `organization` and `organ` to `organ`. These words have different meanings, so a query about the *universe* also retrieves documents about *universities*: more non-relevant documents are returned, which lowers precision. (c) The WordNet lemmatizer needs the part of speech. The default is noun, and `searching` is a valid noun (a gerund), so it is left unchanged; with `pos="v"` it is the *-ing* form of the verb *search* and is reduced to it. (d) Porter and Snowball disagree on `generously` (`gener` vs `generous`) and on `was` (`wa` vs `was`).

### 2.4 · Your `preprocess` function (7 pts)
Write `preprocess(text, stemmer="porter", remove_stop=True)` that runs the whole pipeline with the functions you wrote above:
`tokenize` → `normalize` → (optional) stop-word removal → stemming (`"porter"`, `"snowball"`, `"lemma"` or `None`), and returns the list of terms.

Later TPs start from this function, so keep it clean: you will copy this cell (possibly updated) into the next notebooks.

**Check:** `preprocess(DOCS[9])` → `['stem', 'map', 'search', 'search', 'search', 'one', 'term']`

In [9]:
def preprocess(text, stemmer="porter", remove_stop=True):
    terms = normalize(tokenize(text))
    if remove_stop:
        terms = [t for t in terms if t not in STOP]
    if stemmer == "porter":
        terms = [porter.stem(t) for t in terms]
    elif stemmer == "snowball":
        terms = [snowball.stem(t) for t in terms]
    elif stemmer == "lemma":
        terms = [wnl.lemmatize(t, pos="v") for t in terms]   # verb lemmas: merges searching/searched/searches
    return terms

for i in (0, 4, 9):
    print(preprocess(DOCS[i]))
print(preprocess(DOCS[9], stemmer="lemma"))
print(preprocess(DOCS[9], stemmer=None, remove_stop=False))

['search', 'engin', 'index', 'billion', 'web', 'page', 'everi', 'day']
['googl', 'crawler', 'found', '3', 'new', 'page', 'usa', 'yesterday']
['stem', 'map', 'search', 'search', 'search', 'one', 'term']
['stem', 'map', 'search', 'search', 'search', 'one', 'term']
['stemming', 'maps', 'searching', 'searched', 'and', 'searches', 'to', 'one', 'term']


**Question 2.4.** Compare the Porter and lemma outputs for document 9. Which one would let the query *search* match this document best, and why?

**Answer 2.4:** For document 9 both give `['stem', 'map', 'search', 'search', 'search', 'one', 'term']`: *searching*, *searched* and *searches* are all reduced to `search`, so the query *search* matches the document with a term frequency of 3. Without any reduction (`stemmer=None`) the document would contain `searching`, `searched` and `searches` but not `search`, and the query would miss it. Porter is simpler and needs no part-of-speech information, but produces non-words (`engin`); the lemmatizer gives real words, but only if the right POS is given (with the noun default, `searching` would stay unchanged).

## Exercise 3 · Subword tokenization (10 pts)
Neural models do not use a fixed list of words. They split text into **subwords** learned from data: frequent words stay whole, rare words are split into frequent pieces, so there are no unknown words. BERT uses the **WordPiece** algorithm; GPT models use **BPE** (byte-pair encoding).

### 3.1 · Load a pretrained tokenizer (4 pts)
Check that you obtain `['i', 'have', 'a', 'new', 'gp', '##u', '!']`.

In [10]:
from transformers import AutoTokenizer
bert = AutoTokenizer.from_pretrained("bert-base-uncased")
print(bert.tokenize("I have a new GPU!"))

['i', 'have', 'a', 'new', 'gp', '##u', '!']


### 3.2 · Explore (6 pts)
Tokenize each word of the list and document 4. For each, print the subword tokens and their number. Then add **three words of your own** (e.g. a rare word, a misspelling, a place name).

In [11]:
TEST_WORDS = ["search", "searching", "tokenization", "crawler", "bigram", "perplexity", "serach"]
TEST_WORDS += ["antidisestablishmentarianism", "Battambang", "recieve"]   # my own: rare word, place name, misspelling
for w in TEST_WORDS:
    pieces = bert.tokenize(w)
    print(f"{w:32s} {pieces}  ({len(pieces)})")

bert_toks = bert.tokenize(DOCS[4])
print(bert_toks, len(bert_toks))
print(tokenize(DOCS[4]), len(tokenize(DOCS[4])))

search                           ['search']  (1)
searching                        ['searching']  (1)
tokenization                     ['token', '##ization']  (2)
crawler                          ['crawl', '##er']  (2)
bigram                           ['big', '##ram']  (2)
perplexity                       ['per', '##plex', '##ity']  (3)
serach                           ['sera', '##ch']  (2)
antidisestablishmentarianism     ['anti', '##dis', '##est', '##ab', '##lish', '##ment', '##arian', '##ism']  (8)
Battambang                       ['bat', '##tam', '##bang']  (3)
recieve                          ['rec', '##ie', '##ve']  (3)
['google', "'", 's', 'crawl', '##er', 'found', '3', 'new', 'pages', 'in', 'the', 'u', '.', 's', '.', 'a', '.', 'yesterday', '.'] 19
["Google's", 'crawler', 'found', '3', 'new', 'pages', 'in', 'the', 'U.S.A.', 'yesterday', '.'] 11


**Question 3.2.** (a) What does the prefix `##` mean? (b) Which kind of words are split into several pieces, and which stay whole? Why? (c) Why can a subword tokenizer never produce an *unknown word*?

**Answer 3.2:** (a) `##` marks a piece that continues the previous one inside the same word (not the start of a new word): `gp`, `##u` are the two pieces of *gpu*. (b) Frequent words (*search*, *the*, *pages*) are in the vocabulary and stay whole. Rare words, technical terms, names and misspellings (*tokenization*, *perplexity*, *serach*) are cut into several smaller frequent pieces, because the vocabulary has a fixed size and is learned from data, so it cannot hold every word. (c) The vocabulary contains all single characters (and a special `[UNK]` only for characters outside it), so any word can always be spelled by falling back on smaller pieces; there is no out-of-vocabulary *word*, unlike with a fixed word list. In `DOCS[4]`, BERT produces more tokens than the regex tokenizer because of the split words (and *google's* becomes several pieces).

## Exercise 4 · Bag-of-words and the inverted index (20 pts)
In the **bag-of-words** model a document is represented by the counts of its terms; word order is ignored.

### 4.1 · Document-term matrix from scratch (8 pts)
1. Preprocess every document with your `preprocess` (Porter, stop words removed).
2. Build the vocabulary: the **sorted** list of all distinct terms.
3. Build the matrix `M` (a list of lists): `M[i][j]` = number of times term `vocab[j]` occurs in document `i`.
4. Print the shape, the percentage of zero cells, and the non-zero entries of document 0.

**Check:** the matrix has 12 rows and 48 columns.

In [12]:
processed = [preprocess(d) for d in DOCS]
vocab = sorted({t for p in processed for t in p})
M = [[Counter(p)[t] for t in vocab] for p in processed]

zeros = 100 * sum(c == 0 for row in M for c in row) / (len(M) * len(vocab))
print("shape:", len(M), "x", len(vocab), f"| zeros: {zeros:.1f}%")
print({vocab[j]: c for j, c in enumerate(M[0]) if c > 0})

shape: 12 x 48 | zeros: 87.2%
{'billion': 1, 'day': 1, 'engin': 1, 'everi': 1, 'index': 1, 'page': 1, 'search': 1, 'web': 1}


### 4.2 · Check with scikit-learn (4 pts)
`CountVectorizer` builds the same matrix. Give it your own pipeline with `analyzer=preprocess`, then check that the vocabulary and the matrix are identical to yours.

In [13]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
cv = CountVectorizer(analyzer=preprocess)
X = cv.fit_transform(DOCS)
print("same vocabulary:", list(cv.get_feature_names_out()) == vocab)
print("same matrix:", np.array_equal(X.toarray(), np.array(M)))

same vocabulary: True
same matrix: True


### 4.3 · From the matrix to an inverted index (5 pts)
Build a dictionary `index` that maps each term to its postings list `[(doc_id, tf), ...]`, reading **one column** of `M` per term and keeping only the non-zero cells. Print the postings of `search`, `index`, `page` and `crawler`.

In [14]:
index = {}
for j, term in enumerate(vocab):
    index[term] = [(i, M[i][j]) for i in range(len(M)) if M[i][j] > 0]   # one column of M

for t in ["search", "index", "page", "crawler"]:
    print(t, "->", index[t])

search -> [(0, 1), (1, 1), (2, 1), (5, 1), (9, 3)]
index -> [(0, 1), (1, 1), (5, 2), (6, 1), (8, 1)]
page -> [(0, 1), (1, 1), (3, 1), (4, 1), (7, 1)]
crawler -> [(3, 1), (4, 1)]


**Question 4.3.** (a) Why does scikit-learn return a *sparse* matrix instead of a normal list of lists? (b) What is the relation between a column of `M` and a postings list?

**Answer 4.3:** (a) Most cells are 0 (87% here, and over 99% with a real vocabulary of 100,000+ terms). A sparse matrix stores only the non-zero cells (position and value), so it uses far less memory and makes operations faster. (b) Column *j* of `M` holds the frequency of term *j* in every document. Its non-zero entries, as `(doc_id, tf)` pairs, are exactly the postings list of that term. An inverted index is therefore the sparse, column-wise (transposed) view of the document-term matrix.

### 4.4 · The limit of bag-of-words (3 pts)
Write **two short sentences with different meanings** that get exactly the same bag-of-words vector, and show it with code.

In [15]:
s1, s2 = "dog bites man", "man bites dog"
print(Counter(preprocess(s1)) == Counter(preprocess(s2)))

True


**Answer 4.4:** *"dog bites man"* and *"man bites dog"* have opposite meanings but contain the same terms with the same counts, so their bag-of-words vectors are identical (the code prints `True`). Bag-of-words loses word order, and so who does what to whom. N-grams (Exercise 5) recover some local order.

## Exercise 5 · A bigram language model (30 pts)
A bigram model estimates P(w | previous word) from counts and multiplies these probabilities along a sentence. For language models we **keep stop words and do not stem**: function words matter for word order.

### 5.1 · Counting (6 pts)
* `lm_tokens(text)`: the lower-cased word tokens of `text` (use `tokenize`, drop pure punctuation).
* Training sentences: each document becomes `["<s>"] + lm_tokens(doc) + ["</s>"]`.
* Count bigrams in `bigram_c`, and in `context_c` count each word **as the first word of a bigram** (every token except the last `</s>`).
* `V` = number of distinct word types in the training sentences, **including `</s>` but not `<s>`**.

**Check:** V = 78, 122 bigram types, C(&lt;s&gt;) = 12.

In [16]:
def lm_tokens(text):
    return [t.lower() for t in tokenize(text) if re.search(r"\w", t)]

train = [["<s>"] + lm_tokens(d) + ["</s>"] for d in DOCS]
bigram_c, context_c = Counter(), Counter()
for s in train:
    for prev, w in zip(s, s[1:]):
        bigram_c[(prev, w)] += 1
        context_c[prev] += 1          # prev as first word of a bigram
V = len({w for s in train for w in s if w != "<s>"})
print("V =", V, "| bigram types:", len(bigram_c), "| C(<s>) =", context_c["<s>"])
print(train[3])

V = 78 | bigram types: 122 | C(<s>) = 12
['<s>', 'the', 'crawler', 'downloads', 'web', 'pages', 'and', 'follows', 'their', 'links', '</s>']


### 5.2 · MLE probabilities (6 pts)
P_MLE(w | prev) = C(prev w) / C(prev). Write `p_mle(w, prev)` and `sentence_prob(text, p)`, which pads the sentence with `<s>` and `</s>` and multiplies the bigram probabilities given by the function `p`. Compute the probability of the three sentences.

In [17]:
def p_mle(w, prev):
    return bigram_c[(prev, w)] / context_c[prev] if context_c[prev] > 0 else 0.0

def sentence_prob(text, p):
    s = ["<s>"] + lm_tokens(text) + ["</s>"]
    prob = 1.0
    for prev, w in zip(s, s[1:]):
        prob *= p(w, prev)
    return prob

for s in ["the crawler downloads web pages",
          "the crawler downloads web pages every day",
          "pages web downloads crawler the"]:
    print(f"{sentence_prob(s, p_mle):.6f}  {s}")

0.001667  the crawler downloads web pages
0.000556  the crawler downloads web pages every day
0.000000  pages web downloads crawler the


**Question 5.2.** The second sentence is not in the collection. Why is its probability not zero? Why is the third one zero although it uses the same words as the first?

**Answer 5.2:** The second sentence is not a document of the collection, but each of its bigrams occurs in the training data (`pages every`, `every day`, `day </s>` come from document 0), so every factor is non-zero and the product is positive. The third sentence uses the same words, but the bigram `pages web` never occurs (and neither do `web downloads`, `downloads crawler`, ...), so one factor is 0 and the whole product is 0. The bigram model captures word order, and MLE gives zero to every unseen bigram.

### 5.3 · Add-k smoothing (6 pts)
P_add-k(w | prev) = (C(prev w) + k) / (C(prev) + k·V). Write `p_addk(w, prev, k)` and check that P(engine | search) = 2/81 ≈ 0.0247 for k = 1.

In [18]:
def p_addk(w, prev, k):
    return (bigram_c[(prev, w)] + k) / (context_c[prev] + k * V)

print(p_mle("engine", "search"), p_addk("engine", "search", 1), p_addk("engine", "search", 0.1))
print(sentence_prob("pages web downloads crawler the", lambda w, prev: p_addk(w, prev, 1)))

0.3333333333333333 0.024691358024691357 0.10185185185185186
3.0087746459562845e-12


**Question 5.3.** P(engine | search) is 1/3 with MLE but only 0.025 with k = 1. Where did the probability go?

**Answer 5.3:** Add-k takes probability mass from the seen bigrams and shares it among **all** V = 78 possible next words, including those never seen after `search`. With k = 1, the denominator grows from C(search) = 3 to 3 + 78 = 81, and each of the 78 words gets at least 1/81. So `engine` goes from 1/3 to 2/81 ≈ 0.025, and the mass lost is given to unseen bigrams, which makes the zero-probability problem disappear (the third sentence now has a small non-zero probability).

### 5.4 · Perplexity on held-out sentences (8 pts)
PP = exp(−(1/N) Σ log P(w | prev)), where N counts every predicted token, including `</s>`. Use **log probabilities** (sum of logs) to avoid underflow; return `float("inf")` if a probability is 0.

Compute the perplexity of the held-out sentences with MLE and with add-k for k ∈ {1, 0.5, 0.1, 0.05, 0.01, 0.001}, and show the results in a table.

In [19]:
HELD_OUT = ["the search engine ranks pages for users",
            "users click the top pages",
            "the crawler follows links to new pages"]

def perplexity(sentences, p):
    log_sum, n = 0.0, 0
    for sent in sentences:
        s = ["<s>"] + lm_tokens(sent) + ["</s>"]
        for prev, w in zip(s, s[1:]):
            pr = p(w, prev)
            if pr == 0:
                return float("inf")
            log_sum += math.log(pr)
            n += 1
    return math.exp(-log_sum / n)

print("MLE:", perplexity(HELD_OUT, p_mle))
ks = [1, 0.5, 0.1, 0.05, 0.01, 0.001]
pd.DataFrame({"k": ks,
              "perplexity": [perplexity(HELD_OUT, lambda w, prev, k=k: p_addk(w, prev, k)) for k in ks]})

MLE: inf


,k,perplexity
0,1.000,53.448046
1,0.500,44.414988
2,0.100,28.984635
3,0.050,26.133733
4,0.010,28.145226
5,0.001,56.325069


**Question 5.4.** (a) Why is the MLE perplexity infinite? (b) Which k gives the lowest perplexity? (c) Explain why a **large** k and a **very small** k are both worse.

**Answer 5.4:** (a) The held-out sentences contain bigrams never seen in training (`ranks pages`, `for users`, `users click`, `the top`, `top pages`, `crawler follows`, ...). MLE gives them probability 0, so log P = −∞ and PP = ∞. (b) Among the tested values, k = 0.05 gives the lowest perplexity (≈ 26.1; k = 0.01 gives ≈ 28.1 and k = 0.1 ≈ 29.0). (c) With a **large** k (1 or more) too much mass is moved to unseen bigrams: the model flattens toward a uniform distribution over V words and ignores the counts, so seen bigrams are under-estimated (PP ≈ 53 for k = 1). With a **very small** k (0.001) unseen bigrams get an almost-zero probability, so each of them costs a huge −log P, and the model behaves like MLE (PP ≈ 56, and ∞ as k → 0). The best k is a compromise. (k was chosen on the held-out set itself here; with more data a separate validation set should be used.)